In [0]:
def validate_current_state(df):
    """
    Validates the current state table before loading to the domain history.
    """
    if df.filter(F.col("domain").isNull()).limit(1).count():
        raise ValueError("Current state contains a null domain")

    if (
        df.groupBy("domain").count()
        .filter(F.col("count") > 1)
        .limit(1).count()
    ):
        raise ValueError("Current state contains duplicate domains")

    # Prevent accidental closure of the entire history.
    if df.limit(1).count() == 0:
        raise ValueError("Current state is empty")

    print("Validated current state. No errors found")

In [0]:
def validate_parsed_count(
    parsed_df,
    run_id,
    run_date,
    audit_table,
    max_drop_ratio=0.70,
    expected_feeds=["feed_a", "feed_b", "feed_c", "feed_d"],
):
    """
    Group the input DataFrame by feed_name and validate each feed.

    max_drop_ratio=0.70 allows a record count decrease of up to 70%.
    expected_feeds optionally includes feeds missing from the DataFrame.
    """

    if not 0 <= max_drop_ratio <= 1:
        raise ValueError("max_drop_ratio must be between 0 and 1")

    # Count records for each feed before applying any filters.
    counts_by_feed = {
        row["feed_name"]: row["count"]
        for row in parsed_df.groupBy("feed_name").count().collect()
    }

    if None in counts_by_feed:
        raise ValueError("DataFrame contains records with a null feed_name")

    # Include expected feeds even if they contain no parsed records.
    feeds_to_validate = set(counts_by_feed)

    if expected_feeds is not None:
        feeds_to_validate.update(expected_feeds)

    if not feeds_to_validate:
        raise ValueError(
            "DataFrame is empty and no expected feeds were provided"
        )

    validation_errors = []

    for feed_name in sorted(feeds_to_validate):
        current_count = counts_by_feed.get(feed_name, 0)

        # Retrieve the previous accepted count for this specific feed.
        previous_count = get_previous_parsed_count(
            audit_table=audit_table,
            feed_name=feed_name,
            run_id=run_id,
        )

        drop_ratio = None
        errors = []

        # Validation 1.
        # check that the feed contains at least one parsed record.
        if current_count == 0:
            errors.append("Parsed feed is empty or missing")

        # Validation 2.
        # compare against the previous accepted count for the same feed
        if previous_count is not None and previous_count > 0:
            drop_ratio = (
                previous_count - current_count
            ) / previous_count

            if drop_ratio > max_drop_ratio:
                errors.append(
                    f"Record count dropped by {drop_ratio:.2%}; "
                    f"maximum allowed drop is {max_drop_ratio:.2%}. "
                    f"Previous: {previous_count}, current: {current_count}"
                )

        status = "failed" if errors else "logged"

        # each feed independently, including failed validations.
        log_record_count(
            audit_table=audit_table,
            run_id=run_id,
            run_date=run_date,
            feed_name=feed_name,
            stage="parsed",
            record_count=current_count,
            previous_count=previous_count,
            ratio=drop_ratio,
            status=status,
        )

        counts_by_feed[feed_name] = current_count

        if errors:
            validation_errors.append(f"{feed_name}: " + ", ".join(errors))

    # raise only after all feeds have been validated and saved to audit_table
    if validation_errors:
        raise ValueError(
            "Feed validation failed:\n" + "\n".join(validation_errors)
        )

    print("Feed validation passed. Parsed feeds contain following counts:")

    return counts_by_feed